# Step 2 : FinBERT-First Sampling
**Goal:** Run raw Layer 2 text through FinBERT on a 200-row sample (50 per source)  
**First:** Observe what breaks BEFORE writing any cleaning code  
**Output:** Diagnostic analysis + informed cleaning checklist per source

## 0. Install & Imports

In [2]:
import pandas as pd
import numpy as np
import torch
import warnings
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax
from collections import defaultdict

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cpu


import os
import sys
import re
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

load_dotenv()

# Allow src/ imports without pip install -e .
sys.path.insert(0, os.path.abspath('..'))

from src.sentiment.finbert import load_finbert, run_finbert_batch

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_colwidth', 120)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [3]:
# ── UPDATE THIS PATH to your Layer 2 Parquet ──
LAYER2_PATH = '/kaggle/input/datasets/leev75/layer2/layer2_unified_final.parquet'

df = pd.read_parquet(LAYER2_PATH)

# Confirm expected schema
EXPECTED_COLS = {'doc_id', 'published_at', 'source', 'text', 'ticker', 'url'}
assert EXPECTED_COLS.issubset(df.columns), f'Missing columns: {EXPECTED_COLS - set(df.columns)}'

print(f'Total rows: {len(df):,}')
print(f'Sources: {df["source"].value_counts().to_dict()}')
print(f'Null text: {df["text"].isna().sum()}')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/leev75/layer2/layer2_unified_final.parquet'

LAYER2_PATH = os.getenv('LAYER2_PATH', '../data/raw/layer2/layer2_unified_final.parquet')

df = pd.read_parquet(LAYER2_PATH)

EXPECTED_COLS = {'doc_id', 'published_at', 'source', 'text', 'ticker', 'url'}
assert EXPECTED_COLS.issubset(df.columns), f'Missing columns: {EXPECTED_COLS - set(df.columns)}'

print(f'Total rows: {len(df):,}')
print(f'Sources: {df["source"].value_counts().to_dict()}')
print(f'Null text: {df["text"].isna().sum()}')

In [ ]:
SAMPLE_PER_SOURCE = 50
RANDOM_SEED = 42

# Drop null text before sampling
df_valid = df.dropna(subset=['text']).copy()

df_valid['text'] = df_valid['text'].astype(str).str.strip()
df_valid = df_valid[df_valid['text'].str.len() > 0]

# Stratified sample
sample_frames = []
for src, grp in df_valid.groupby('source'):
    n = min(SAMPLE_PER_SOURCE, len(grp))
    sample_frames.append(grp.sample(n=n, random_state=RANDOM_SEED))

sample = pd.concat(sample_frames).reset_index(drop=True)

print(f'Sample size: {len(sample)}')
print(sample['source'].value_counts())

Sample size: 200
source
news               50
reddit             50
twitter_general    50
twitter_musk       50
Name: count, dtype: int64


## 3. Load FinBERT — RAW, No Cleaning Yet

In [ ]:
MODEL_NAME = 'ProsusAI/finbert'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

# Label order from FinBERT config: positive=0, negative=1, neutral=2
# Verify:
print('Label map:', model.config.id2label)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Label map: {0: 'positive', 1: 'negative', 2: 'neutral'}


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer, model = load_finbert(device=DEVICE)
# Label order from FinBERT config: positive=0, negative=1, neutral=2

In [ ]:
# ── Tokenize WITHOUT truncation to see raw token lengths ──
diag_rows = []

for _, row in sample.iterrows():
    tokens = tokenizer.encode(row['text'], add_special_tokens=True)
    token_len = len(tokens)
    
    diag_rows.append({
        'doc_id':       row['doc_id'],
        'source':       row['source'],
        'char_len':     len(row['text']),
        'token_len':    token_len,
        'exceeds_512':  token_len > 512,
        'text_preview': row['text'][:120]
    })

diag_df = pd.DataFrame(diag_rows)

print('=== Token Length Stats by Source ===')
print(diag_df.groupby('source')['token_len'].describe().round(1))
print()
print('=== Rows Exceeding 512 Tokens ===')
print(diag_df.groupby('source')['exceeds_512'].sum())

Token indices sequence length is longer than the specified maximum sequence length for this model (1627 > 512). Running this sequence through the model will result in indexing errors


=== Token Length Stats by Source ===
                 count    mean     std  min   25%    50%     75%      max
source                                                                   
news              50.0  1139.0  2308.4  4.0  40.8  494.5  1129.8  14277.0
reddit            50.0   287.3   558.0  4.0  25.2  101.0   266.2   3077.0
twitter_general   50.0    53.5    29.2  8.0  30.5   48.0    77.2    125.0
twitter_musk      50.0    30.9    20.0  6.0  15.5   25.5    41.0     75.0

=== Rows Exceeding 512 Tokens ===
source
news               25
reddit              5
twitter_general     0
twitter_musk        0
Name: exceeds_512, dtype: int64


In [ ]:
# ── Inspect longest texts per source ──
print('=== Top 3 Longest Texts Per Source ===')
for src in diag_df['source'].unique():
    subset = diag_df[diag_df['source'] == src].nlargest(3, 'token_len')
    print(f'\n--- {src} ---')
    print(subset[['token_len', 'text_preview']].to_string(index=False))

=== Top 3 Longest Texts Per Source ===

--- news ---
 token_len                                                                                                               text_preview
     14277   Corona-Krise greift um sich: McDonald‘s extrem betroffen - Mega-Einbruch im Welthandel. Corona-Krise greift um sich: McD
      6099   Stocks Close Higher, Dow Gains 300 Points as Fed's Bostic Spurs Relief Rally: Live Updates. Stocks rose Thursday in an a
      5652 20 climate change documentaries you need to watch because this planet is NOT fine. > Social Good\n\nIn case you hadn’t not

--- reddit ---
 token_len                                                                                                             text_preview
      3077 Open Discussion! Robinhood Trading Suspensions. Why did Robinhood suspend up to 50 stocks during the Sneeze? We all know
      1959 Debt-Free Millionaire at 31: The Dumb Way. 2 years ago I [posted here](https://www.reddit.com/r/financialindependence/co

## 5. Noise Pattern Scanner — Raw Text

In [ ]:
# ── Detect noise patterns WITHOUT fixing them ──
noise_patterns = {
    'has_url':          r'https?://\S+',
    'has_cashtag':      r'\$[A-Z]{1,5}',
    'has_mention':      r'@\w+',
    'has_hashtag':      r'#\w+',
    'has_emoji':        r'[\U00010000-\U0010ffff\U0001F300-\U0001F9FF]',
    'has_html_tag':     r'<[^>]+>',
    'has_rt_prefix':    r'^RT\s',
    'has_markdown':     r'[\*\_\[\]\#\`]{2,}',
    'has_newlines':     r'\n',
    'has_repeated_char':r'(.)\1{3,}',
    'is_short':         None,  # handled separately
}

noise_results = defaultdict(lambda: defaultdict(int))

for _, row in sample.iterrows():
    src = row['source']
    txt = row['text']
    for pattern_name, pattern in noise_patterns.items():
        if pattern_name == 'is_short':
            if len(txt.split()) < 5:
                noise_results[src][pattern_name] += 1
        elif re.search(pattern, txt):
            noise_results[src][pattern_name] += 1

noise_summary = pd.DataFrame(noise_results).fillna(0).astype(int)
print('=== Noise Pattern Counts by Source (out of 50 each) ===')
print(noise_summary)

=== Noise Pattern Counts by Source (out of 50 each) ===
                   news  reddit  twitter_general  twitter_musk
has_newlines         35      25               19             9
is_short             11       4                1             9
has_mention           2       0               10            42
has_url               1       8               24             7
has_hashtag           1       2                6             0
has_markdown          0       5                0             0
has_repeated_char     0       4                0             0
has_emoji             0       1               11             2
has_cashtag           0       0               36             0
has_rt_prefix         0       0                0             3


## 6. Run FinBERT Inference — First-Chunk Truncation (512)

In [ ]:
def run_finbert_batch(texts, tokenizer, model, device, batch_size=16):
    """
    Run FinBERT on a list of texts.
    Strategy: first-chunk truncation at 512 tokens.
    Returns list of dicts with p_pos, p_neg, p_neu, label, score.
    """
    results = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,       # first-chunk truncation
            max_length=512,
            return_tensors='pt'
        ).to(device)
        
        with torch.no_grad():
            logits = model(**encoded).logits
        
        probs = softmax(logits, dim=1).cpu().numpy()
        
        for p in probs:
            # FinBERT label order: positive=0, negative=1, neutral=2
            label_idx = int(np.argmax(p))
            label_map = {0: 'positive', 1: 'negative', 2: 'neutral'}
            results.append({
                'p_pos':            round(float(p[0]), 4),
                'p_neg':            round(float(p[1]), 4),
                'p_neu':            round(float(p[2]), 4),
                'sentiment_label':  label_map[label_idx],
                'sentiment_score':  round(float(p[0]) - float(p[1]), 4),  # p_pos - p_neg
            })
    
    return results

print('Running FinBERT on raw sample...')
preds = run_finbert_batch(sample['text'].tolist(), tokenizer, model, DEVICE)
preds_df = pd.DataFrame(preds)
results = pd.concat([sample.reset_index(drop=True), preds_df], axis=1)

print(f'Done. Shape: {results.shape}')
results[['doc_id', 'source', 'sentiment_label', 'sentiment_score', 'p_pos', 'p_neg', 'p_neu']].head(10)

Running FinBERT on raw sample...
Done. Shape: (200, 11)


,doc_id,source,sentiment_label,sentiment_score,p_pos,p_neg,p_neu
0,64c013fcd28355ef5e307501f3b4477e,news,neutral,0.0848,0.1068,0.0220,0.8711
1,001e4f997bcfacf27a015714404a34ee,news,neutral,0.0307,0.0619,0.0313,0.9068
2,6465f0c71024f206dd3eadc5ec73fe40,news,neutral,0.0467,0.0724,0.0257,0.9019
3,e70eadb26901bb95a77ce5e7202c0d2e,news,negative,-0.9485,0.0119,0.9604,0.0277
4,6bcc97e1f9f0c879a205f42d4a0f04a7,news,positive,0.8728,0.8986,0.0259,0.0755
5,97c29039eb029199ba74484b92846df5,news,neutral,0.0158,0.0858,0.0701,0.8441
6,573ca1551b39f3e36653b8679f073131,news,neutral,0.0196,0.0531,0.0335,0.9135
7,7ea022249a7e44b8cc55b1e87316e351,news,neutral,0.0563,0.0810,0.0247,0.8944
8,38aa940171b8c915fd4cd5b5709434db,news,neutral,0.0152,0.0470,0.0318,0.9212
9,38d2b781e7fad64aff55c434dd67de09,news,neutral,0.0580,0.0924,0.0344,0.8731


# run_finbert_batch is imported from src.sentiment.finbert (see cell 0)
print('Running FinBERT on raw sample...')
preds = run_finbert_batch(sample['text'].tolist(), tokenizer, model, DEVICE)
preds_df = pd.DataFrame(preds)
results = pd.concat([sample.reset_index(drop=True), preds_df], axis=1)

print(f'Done. Shape: {results.shape}')
results[['doc_id', 'source', 'sentiment_label', 'sentiment_score', 'p_pos', 'p_neg', 'p_neu']].head(10)

In [ ]:
print('=== Sentiment Distribution by Source ===')
print(results.groupby(['source', 'sentiment_label']).size().unstack(fill_value=0))
print()
print('=== Sentiment Score Stats by Source ===')
print(results.groupby('source')['sentiment_score'].describe().round(3))

=== Sentiment Distribution by Source ===
sentiment_label  negative  neutral  positive
source                                      
news                    7       38         5
reddit                  8       37         5
twitter_general         8       35         7
twitter_musk            5       42         3

=== Sentiment Score Stats by Source ===
                 count   mean    std    min    25%    50%    75%    max
source                                                                 
news              50.0 -0.028  0.391 -0.948 -0.024  0.019  0.056  0.873
reddit            50.0 -0.056  0.380 -0.952 -0.237 -0.019  0.064  0.895
twitter_general   50.0 -0.008  0.396 -0.938 -0.043  0.024  0.147  0.667
twitter_musk      50.0  0.035  0.343 -0.870 -0.004  0.030  0.096  0.927


In [ ]:
# ── Flag low-confidence predictions (max prob < 0.6) ──
results['max_prob'] = results[['p_pos', 'p_neg', 'p_neu']].max(axis=1)
results['low_confidence'] = results['max_prob'] < 0.60

print('=== Low-Confidence Predictions by Source ===')
print(results.groupby('source')['low_confidence'].sum())
print()
print('=== Sample Low-Confidence Rows ===')
low_conf = results[results['low_confidence']]
print(low_conf[['source', 'sentiment_label', 'max_prob', 'text']].head(10).to_string())

=== Low-Confidence Predictions by Source ===
source
news               0
reddit             8
twitter_general    3
twitter_musk       2
Name: low_confidence, dtype: int64

=== Sample Low-Confidence Rows ===
              source sentiment_label  max_prob                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [ ]:
# ── Spot-check: neutral predictions — are they genuinely neutral or noise? ──
neutrals = results[results['sentiment_label'] == 'neutral'].sample(min(10, len(results[results['sentiment_label'] == 'neutral'])), random_state=42)
print('=== 10 Random Neutral Predictions ===')
for _, row in neutrals.iterrows():
    print(f"[{row['source']}] score={row['sentiment_score']:.3f} | {row['text'][:150]}")
    print()

=== 10 Random Neutral Predictions ===
[reddit] score=-0.426 | If the stock tanks to levels unseen before. Could the board of Tesla kick Elon out? Or Elon has total control and he’s the board.. I love Tesla and ev

[twitter_musk] score=0.009 | @we5dogg @teslaownersSV No escaping that for me

[twitter_general] score=-0.007 | I work with professional portfolio managers for a living.

I have yet to talk to a single one who believes Elon's guidance on FSD. Including PMs who a

[twitter_general] score=0.057 | In January I posted my conviction rankings for 2021 stocks in my portfolio on @JoinCommonstock:

Top 6 in order w/ YTD returns
$MSFT 49%
$AMZN 6%
$TSL

[twitter_musk] score=0.060 | @Teslaconomics That’s the goal!

[twitter_musk] score=-0.258 | @Carnage4Life I sure hope that’s not true at Tesla or SpaceX!

[news] score=-0.024 | 涨幅超过650%，接连被减持，机构“恐高”宁德时代？. 特别声明：以上内容(如有图片或视频亦包括在内)为自媒体平台“网易号”用户上传并发布，本平台仅提供信息存储服务。

Notice: The content above (including the pictures and videos if a

[news] sco

## 8. Cleaning Checklist — Auto-Generated from Diagnostics

In [ ]:
# ── Build a per-source cleaning priority checklist ──
print('=' * 60)
print('CLEANING CHECKLIST — Derived from Day 1 Diagnostics')
print('=' * 60)

THRESHOLD = 5  # flag if pattern appears in >5 of 50 rows

cleaning_tasks = {
    'has_url':          'Strip URLs',
    'has_cashtag':      'Handle cashtags ($TSLA) — keep or normalize',
    'has_mention':      'Strip @mentions',
    'has_hashtag':      'Strip or normalize #hashtags',
    'has_emoji':        'Remove emojis',
    'has_html_tag':     'Strip HTML tags',
    'has_rt_prefix':    'Remove RT prefix (Twitter)',
    'has_markdown':     'Strip markdown syntax (Reddit)',
    'has_newlines':     'Normalize newlines to spaces',
    'has_repeated_char':'Remove repeated characters',
    'is_short':         'Flag short texts (<5 words) — consider filtering',
}

for src in noise_summary.columns:
    print(f'\n--- {src} ---')
    flagged = False
    for pattern, task in cleaning_tasks.items():
        count = noise_summary.loc[pattern, src] if pattern in noise_summary.index else 0
        if count > THRESHOLD:
            print(f'  [HIGH  {count:>3}/50] {task}')
            flagged = True
        elif count > 0:
            print(f'  [LOW   {count:>3}/50] {task}')
            flagged = True
    if not flagged:
        print('  No major noise patterns detected.')

print()
# Truncation summary
print('--- Truncation Risk ---')
trunc = diag_df.groupby('source')['exceeds_512'].sum()
for src, count in trunc.items():
    if count > 0:
        print(f'  [{src}] {count}/50 texts exceed 512 tokens → first-chunk strategy confirmed')
    else:
        print(f'  [{src}] No truncation needed')

CLEANING CHECKLIST — Derived from Day 1 Diagnostics

--- news ---
  [LOW     1/50] Strip URLs
  [LOW     2/50] Strip @mentions
  [LOW     1/50] Strip or normalize #hashtags
  [HIGH   35/50] Normalize newlines to spaces
  [HIGH   11/50] Flag short texts (<5 words) — consider filtering

--- reddit ---
  [HIGH    8/50] Strip URLs
  [LOW     2/50] Strip or normalize #hashtags
  [LOW     1/50] Remove emojis
  [LOW     5/50] Strip markdown syntax (Reddit)
  [HIGH   25/50] Normalize newlines to spaces
  [LOW     4/50] Remove repeated characters
  [LOW     4/50] Flag short texts (<5 words) — consider filtering

--- twitter_general ---
  [HIGH   24/50] Strip URLs
  [HIGH   36/50] Handle cashtags ($TSLA) — keep or normalize
  [HIGH   10/50] Strip @mentions
  [HIGH    6/50] Strip or normalize #hashtags
  [HIGH   11/50] Remove emojis
  [HIGH   19/50] Normalize newlines to spaces
  [LOW     1/50] Flag short texts (<5 words) — consider filtering

--- twitter_musk ---
  [HIGH    7/50] Strip URLs
  [H

## 9. Save Sample Results

In [ ]:
OUTPUT_PATH = 'day1_finbert_sample_results.parquet'

save_cols = ['doc_id', 'published_at', 'source', 'text',
             'p_pos', 'p_neg', 'p_neu', 'sentiment_label',
             'sentiment_score', 'max_prob', 'low_confidence']

results[save_cols].to_parquet(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH}')
print(f'Rows: {len(results)}')

Saved: day1_finbert_sample_results.parquet
Rows: 200


OUTPUT_PATH = os.path.join(
    os.getenv('OUTPUT_PATH', '../outputs'),
    'data',
    'step2_finbert_sample_results.parquet'
)
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

save_cols = ['doc_id', 'published_at', 'source', 'text',
             'p_pos', 'p_neg', 'p_neu', 'sentiment_label',
             'sentiment_score', 'max_prob', 'low_confidence']

results[save_cols].to_parquet(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH}')
print(f'Rows: {len(results)}')